In [1]:
"""
HiLo Deviated Strategy EV Simulation (Monte Carlo Averaged)

For each shoe state:
1. Play one round on original shoe (advances shoe state for next iteration)
2. Run N Monte Carlo simulations using copy_reset_sampler (each with fresh RNG)
3. Record average EV from MC runs as the expected value estimate

This produces much cleaner EV estimates compared to single-round outcomes,
since averaging over many runs reduces variance from ~$115 to ~$115/sqrt(N).

Results are appended to CSV file for incremental data collection.
"""

import numpy as np
import pandas as pd
import os
import random
import time
import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

from blackjack.rules import BJRules
from blackjack_cpp import ProbabilisticRankShoe, load_combo_data
from blackjack import count_strategy_play
from strategy import CardCounter, DeviatedBasicStrategy


In [2]:
# Load combo data for the game tree
current_dir = os.getcwd()
load_combo_data(os.path.join(current_dir, "../combinations/s16_new/"), 11)


In [3]:
# Game rules configuration (same as treewalker_strategy.ipynb)
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=True,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3 / 2,
    surrender_payout=1 / 2,
    insurance_payout=2 / 1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

# Initialize deviated basic strategy
dev_strategy = DeviatedBasicStrategy(
    basic_strategy_folder="../strategy",
    deviation_strategy_folder="../strategy"
)


In [4]:
def simulate_shoe(seed, n_mc_runs=50):
    """
    Simulate a full shoe and collect EV data for each round.
    
    For each round position:
    1. Record shoe state
    2. Play one round on original shoe (this advances the shoe state)
    3. Run n_mc_runs Monte Carlo simulations using copy_reset_sampler
    4. Record average EV from MC runs
    
    Returns list of dicts with:
    - tc: integer true count
    - tc_float: float true count
    - ev: average player value from Monte Carlo runs ($100 round)
    - rank_count: dict of remaining card counts before the round
    """
    shoe = ProbabilisticRankShoe(n_decks=6, seed=seed)
    counter = CardCounter(n_decks=6)
    
    bet_unit = 100  # $100 round
    results = []
    
    # Play until less than 52 cards remain (1 deck)
    while sum(shoe.get_rank_count().values()) >= 52:
        # Record state BEFORE the round
        tc_float = counter.get_true_count()
        tc_int = counter.get_integer_tc()
        rank_count = shoe.get_rank_count().copy()
        
        # Run Monte Carlo simulations to estimate EV
        mc_values = []
        for _ in range(n_mc_runs):
            # Create a copy with reset sampler (fresh RNG for each MC run)
            shoe_copy = shoe.copy_reset_sampler()
            counter_copy = CardCounter.from_rank_count(rank_count, n_decks=6)
            
            # Play round on copy
            end_round_mc = count_strategy_play.play_round(
                rules, shoe_copy, bet_unit, counter_copy, dev_strategy
            )
            mc_values.append(end_round_mc.get_player_value())
        
        # Average EV from MC runs
        ev_estimate = sum(mc_values) / len(mc_values)
        
        # Now play one round on original shoe to advance it
        end_round = count_strategy_play.play_round(
            rules, shoe, bet_unit, counter, dev_strategy
        )
        
        results.append({
            "tc": tc_int,
            "tc_float": tc_float,
            "ev": ev_estimate,
            "rank_count": rank_count
        })
    
    return results


In [5]:
# Simulation parameters
n_shoes = 1000  # Reduced since each shoe now runs 50 MC simulations per round
n_mc_runs = 100   # Number of Monte Carlo runs per shoe state
n_workers = multiprocessing.cpu_count() - 1

# Generate random seeds
seed = int(time.time()) % 1234567890
print(f"Master seed: {seed}")
random.seed(seed)
seeds = [random.randrange(10000000) for _ in range(n_shoes)]

print(f"Simulating {n_shoes} shoes with {n_workers} workers...")
print(f"Each round uses {n_mc_runs} Monte Carlo runs to estimate EV")


Master seed: 531601381
Simulating 1000 shoes with 15 workers...
Each round uses 100 Monte Carlo runs to estimate EV


In [ ]:
# Run parallel simulation
from functools import partial

all_results = []
shoe_results = []
simulate_fn = partial(simulate_shoe, n_mc_runs=n_mc_runs)

with ProcessPoolExecutor(max_workers=n_workers) as executor: 
    for result in tqdm.tqdm(
        executor.map(simulate_fn, seeds), 
        total=n_shoes
    ):
        shoe_results.append(result)


 69%|██████▉   | 693/1000 [44:34<19:44,  3.86s/it]  


KeyboardInterrupt: 

In [ ]:
# Flatten results
for shoe_result in shoe_results:
    all_results.extend(shoe_result)

print(f"Collected {len(all_results)} rounds from {n_shoes} shoes")

NameError: name 'shoe_results' is not defined

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

# Expand rank_count dict to individual columns (n2, n3, ..., n10, nA)
for r in range(2, 12):
    col_name = f"n{r}" if r < 11 else "nA"
    df[col_name] = df["rank_count"].apply(lambda x: x[r])

# Drop the rank_count column
df = df.drop(columns=["rank_count"])

# Reorder columns to match true_edge_vs_count_best.csv format (without ev_min/ev_max)
column_order = ["tc", "tc_float", "ev"] + [f"n{r}" for r in range(2, 11)] + ["nA"]
df = df[column_order]

print(f"DataFrame shape: {df.shape}")
df.head()


In [ ]:
# Save to CSV (append if exists, create if not)
# New filename for MC-averaged EV data
csv_path = "hilo_ev_vs_count_mc.csv"

if os.path.exists(csv_path):
    # Append without header
    df.to_csv(csv_path, mode='a', header=False, index=False)
    print(f"Appended {len(df)} rows to existing file '{csv_path}'")
else:
    df.to_csv(csv_path, index=False)
    print(f"Created new file '{csv_path}' with {len(df)} rows")

print(f"Note: EV values are averages of {n_mc_runs} Monte Carlo runs per shoe state")

# Clear results to free memory
all_results = []


In [ ]:
# Load and display accumulated data statistics
csv_path = "hilo_ev_vs_count_mc.csv"  # MC-averaged data
df_all = pd.read_csv(csv_path)
print(f"Total rows in '{csv_path}': {len(df_all)}")
print(f"\nEV statistics (Monte Carlo averaged):")
print(f"  Mean EV: ${df_all['ev'].mean():.4f}")
print(f"  Std EV:  ${df_all['ev'].std():.4f}")
print(f"\nNote: Std should be much lower than single-round data (~$115)")
print(f"      since EV values are averages of {n_mc_runs} runs")
print(f"\nTrue count distribution:")
print(df_all['tc'].value_counts().sort_index())


In [ ]:
# Quick summary: mean EV by integer true count
ev_by_tc = df_all.groupby('tc')['ev'].agg(['mean', 'std', 'count'])
ev_by_tc.columns = ['mean_ev', 'std_ev', 'n_samples']
print("Mean EV ($100 round) by True Count:")
print(ev_by_tc.to_string())
